In [1]:
# fetch standardized commodity price series from FAO API (INTERNATIONAL only)
# and save to CSV

import requests
import pandas as pd
import numpy as np
from time import sleep

# ── CONFIG ─────────────────────────────────────────────────────────────────────
OUTPUT_FILE = "international_commodity_standardized.csv"

# ── UNIT MAPS ──────────────────────────────────────────────────────────────────
UNIT_TO_SINGLE = {
    # ── MASS → kg ──────────────────────────────────────────────────────────────
    'Kg':                        1.0,
    '1 kg':                      1.0,
    '1.1 Kg':                    1.1,
    '1.5 Kg':                    1.5,
    '2 Kg':                      2.0,
    '2 kg':                      2.0,
    '2.5 kg':                    2.5,
    '3 kg':                      3.0,
    '3.5 kg':                    3.5,
    '5 kg':                      5.0,
    '10 Kg':                     10.0,
    '10 kg':                     10.0,
    '12.5 Kg':                   12.5,
    '20 kg':                     20.0,
    '25 kg':                     25.0,
    '30 kg':                     30.0,
    '50 kg':                     50.0,
    '60 kg':                     60.0,
    '100 Kg':                    100.0,
    '100 kg':                    100.0,
    'tonne':                     1000.0,
    '120 kg':                    120.0,
    '160 kg':                    160.0,
    # grams
    '85 gms':                    0.085,
    '100 gms':                   0.1,
    '240 gms':                   0.24,
    '400 gms':                   0.4,
    '500 gms':                   0.5,
    '0.5 kg':                    0.5,
    '700 gms':                   0.7,
    '720 gms':                   0.72,
    '750-800 gms':               0.775,
    '800 gms':                   0.8,
    '810 gms':                   0.81,
    '900 gms':                   0.9,
    '900 gms -1.5kg':            1.2,
    # pounds / libras
    'lb':                        0.4536,
    '6 lbs':                     2.7216,
    '100 lbs':                   45.36,
    'Libra':                     0.4536,
    '3 Libras':                  1.3608,
    '50 Libras':                 22.68,
    # regional weight units
    'Bolivian arroba (11.5 kg)': 11.5,
    'Spanish quintal (46 kg)':   46.0,
    'Box (18.14 kg)':            18.14,
    'Cuartilla (2.88 kg)':       2.88,
    'Coro':                      2.5,
    'pyi (2.13 kg)':             2.13,
    'viss (1.63 kg)':            1.63,
    # ── VOLUME → L ─────────────────────────────────────────────────────────────
    'Liter':                     1.0,
    '0.9 Liter':                 0.9,
    '1.5 Liter':                 1.5,
    '1.8 Liter':                 1.8,
    '2 Liters':                  2.0,
    '5 Liter':                   5.0,
    '100 liters':                100.0,
    '500 ml':                    0.5,
    '750 ml':                    0.75,
    '2 pints':                   0.9464,
    'Gallon':                    3.785,
    'Barrel':                    158.987,
    '20 Liter':                  20.0,
    # ── COUNT → units ───────────────────────────────────────────────────────────
    '1 unit':                    1.0,
    '1 Tray':                    30.0,
    '1/2 Dozen':                 6.0,
    'Dozen':                     12.0,
    '2.5 Dozen':                 30.0,
    '4 loaves':                  4.0,
    '10 units':                  10.0,
    '30 units':                  30.0,
}

UNIT_TO_STD = {
    # mass → kg
    'Kg': 'kg', '1 kg': 'kg', '1.1 Kg': 'kg', '1.5 Kg': 'kg',
    '2 Kg': 'kg', '2 kg': 'kg', '2.5 kg': 'kg', '3 kg': 'kg', '3.5 kg': 'kg',
    '5 kg': 'kg', '10 Kg': 'kg', '10 kg': 'kg', '12.5 Kg': 'kg',
    '20 kg': 'kg', '25 kg': 'kg', '30 kg': 'kg', '50 kg': 'kg',
    '60 kg': 'kg', '100 Kg': 'kg', '100 kg': 'kg', 'tonne': 'kg',
    '85 gms': 'kg', '100 gms': 'kg', '240 gms': 'kg', '400 gms': 'kg',
    '500 gms': 'kg', '0.5 kg': 'kg', '700 gms': 'kg', '720 gms': 'kg',
    '750-800 gms': 'kg', '800 gms': 'kg', '900 gms': 'kg', '900 gms -1.5kg': 'kg',
    'lb': 'kg', '6 lbs': 'kg', '100 lbs': 'kg',
    'Libra': 'kg', '3 Libras': 'kg', '50 Libras': 'kg',
    'Bolivian arroba (11.5 kg)': 'kg', 'Spanish quintal (46 kg)': 'kg',
    'Box (18.14 kg)': 'kg', 'Cuartilla (2.88 kg)': 'kg',
    'Coro': 'kg', 'pyi (2.13 kg)': 'kg', 'viss (1.63 kg)': 'kg', '120 kg': 'kg',
    '160 kg': 'kg', '810 gms': 'kg',
    # volume → L
    'Liter': 'L', '0.9 Liter': 'L', '1.5 Liter': 'L', '1.8 Liter': 'L',
    '2 Liters': 'L', '5 Liter': 'L', '100 liters': 'L',
    '500 ml': 'L', '750 ml': 'L', '2 pints': 'L',
    'Gallon': 'L', 'Barrel': 'L', '20 Liter': 'L',
    # count → unit
    '1 unit': 'unit', '1 Tray': 'unit', '1/2 Dozen': 'unit',
    'Dozen': 'unit', '2.5 Dozen': 'unit', '4 loaves': 'unit',
    '10 units': 'unit', '30 units': 'unit',
}


def compute_price_per_unit(price_usd, unit):
    if pd.isna(price_usd) or price_usd is None:
        return np.nan
    divisor = UNIT_TO_SINGLE.get(unit)
    if divisor is None:
        return np.nan
    return round(float(price_usd) / divisor, 6)


def get_unit_std(unit):
    return UNIT_TO_STD.get(unit, None)


def recompute_price_per_unit(df: pd.DataFrame) -> pd.DataFrame:
    """Recompute price_per_unit for any row that now has price_usd."""
    mask = df["price_usd"].notna() & df["price_per_unit"].isna()
    if mask.any():
        df.loc[mask, "price_per_unit"] = df.loc[mask].apply(
            lambda r: compute_price_per_unit(r["price_usd"], r["unit"]), axis=1
        )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# SERIES BUILDER
# ══════════════════════════════════════════════════════════════════════════════

FINAL_COLS = [
    "date", "price_usd", "price_local", "currency", "price_per_unit", "unit_std",
    "commodity_name", "iso3_country_code", "country", "market",
    "price_type", "unit", "price_source", "fill_method"
]


def build_series_df(datapoints, meta: dict, price_source: str) -> pd.DataFrame | None:
    rows = []
    for dp in datapoints:
        price_usd   = dp.get("price_value_dollar")
        price_local = dp.get("price_value")
        if price_local is None:
            price_local = dp.get("price_value_nominal")

        if price_usd is None and price_local is None:
            continue

        rows.append({
            "date":              dp["date"],
            "price_usd":         price_usd,
            "price_local":       price_local,
            "currency":          meta["currency"],
            "price_per_unit":    compute_price_per_unit(price_usd, meta["unit"]),
            "unit_std":          get_unit_std(meta["unit"]),
            "commodity_name":    meta["commodity_name"],
            "iso3_country_code": meta["iso3"],
            "country":           meta["country"],
            "market":            meta["market"],
            "price_type":        meta["price_type"],
            "unit":              meta["unit"],
            "price_source":      price_source,
            "fill_method":       "original",
        })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])

    # Normalize month-end dates to month-start
    df["date"] = df["date"].dt.to_period("M").dt.to_timestamp()

    # Deduplicate: keep row with most data (fewest NaNs) per month
    df = (df.groupby("date", as_index=False)
            .apply(lambda g: g.loc[g.isna().sum(axis=1).idxmin()], include_groups=False)
            .reset_index(drop=True))

    # Reindex to full monthly range (no gaps)
    full_range = pd.date_range(start=df["date"].min(),
                               end=df["date"].max(), freq="MS")
    df = (df.set_index("date")
            .reindex(full_range)
            .rename_axis("date")
            .reset_index())

    # Re-fill static metadata columns after reindex
    for col in ["commodity_name", "iso3_country_code", "country", "market",
                "price_type", "unit", "unit_std", "currency",
                "price_source", "fill_method"]:
        df[col] = df[col].ffill().bfill()

    df["fill_method"] = df["fill_method"].fillna("original")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# INTERNATIONAL SERIES
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("FETCHING INTERNATIONAL SERIES")
print("=" * 60)

url_intl  = "https://fpma.fao.org/giews/v4/global/price_module/api/v1/FpmaSerieInternational/?limit=500"
intl_data = requests.get(url_intl).json()
all_rows  = []

for item in intl_data["results"]:
    uuid           = item["uuid"]
    commodity_name = item.get("commodity_name", "Unknown")
    country        = item.get("market_name", "Unknown")
    iso3           = item.get("iso3_country_code", "Unknown")
    market         = item.get("country_name", "Unknown")
    price_type     = item.get("price_type", "Unknown")
    unit           = item.get("measure_unit_label", "Unknown")

    url_price = (
        f"https://fpma.fao.org/giews/v4/global/price_module/api/v1/"
        f"FpmaSeriePrice/?uuid__in={uuid}&periodicity=monthly"
    )
    resp = requests.get(url_price).json()
    if resp["count"] == 0:
        sleep(0.2)
        continue

    datapoints = resp["results"][0]["datapoints"]
    meta = dict(commodity_name=commodity_name, iso3=iso3, country=country,
                market=market, price_type=price_type, unit=unit, currency="USD")

    df = build_series_df(datapoints, meta, price_source="International")
    if df is None:
        sleep(0.2)
        continue

    # For international series, price_local IS price_usd (currency=USD)
    mask = df["price_usd"].isna() & df["price_local"].notna()
    df.loc[mask, "price_usd"]   = df.loc[mask, "price_local"]
    df.loc[mask, "fill_method"] = "local_is_usd"

    df = recompute_price_per_unit(df)
    all_rows.append(df[FINAL_COLS])
    print(f"  {commodity_name} — {len(df)} rows")
    sleep(0.2)


# ══════════════════════════════════════════════════════════════════════════════
# SAVE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SAVING")
print("=" * 60)

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    final_df = pd.concat(all_rows, ignore_index=True)

final_df = final_df.sort_values(
    ["country", "market", "commodity_name", "date"]
).reset_index(drop=True)

print(f"\nTotal rows          : {len(final_df):,}")
print(f"Missing price_usd   : {final_df['price_usd'].isna().sum():,}")
print(f"Missing price_local : {final_df['price_local'].isna().sum():,}")
print(f"\nFill method breakdown:")
print(final_df["fill_method"].value_counts().to_string())

final_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved → {OUTPUT_FILE}")

FETCHING INTERNATIONAL SERIES
  Groundnuts (US Runners 40/50, c.i.f. Rotterdam) — 361 rows
  Cassava Chips (Shipments to China, f.o.b. Koh Sichang) — 206 rows
  Diammonium phosphate, DAP (P fertilizer) — 313 rows
  Dairy: Skim Milk Powder (European & Oceania average indicative export prices, f.o.b) — 433 rows
  Fish oil (Any origin, c.i.f. North West Europe) — 361 rows
  Crude oil (Brent) — 313 rows
  Wheat (CWRS, 13.5%) — 314 rows
  Rice (Super Kernel White Basmati 2%) — 313 rows
  Sorghum — 297 rows
  Palmkernel meal (Expeller pellets, 21/23%, c.i.f. Rotterdam) — 361 rows
  Barley (feed) — 309 rows
  Meat: Pig meat (Meat of swine, fresh, chilled or frozen) — 433 rows
  Dairy: Cheddar Cheese (European & Oceania average indicative export prices, f.o.b) — 433 rows
  Wheat (milling, d.a.p. Saryagash station) — 100 rows
  Rice (Parboiled 100%) — 313 rows
  Tea — 226 rows
  Sorghum (US No. 2, Yellow) — 261 rows
  Sunflowerseed (EU, c.i.f. Amsterdam) — 241 rows
  Bananas (Main Brands Centra